In [2]:
import os
import json
import time
import random
import logging
from datasets import load_dataset, Dataset, DatasetDict
from translatepy import Translator
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# Создаём переводчик
yandex = YandexTranslate()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [2]:
# Тест переводчика
test_text = "Hello, world!"
result = yandex.translate(test_text, "ru")
print(f"Original: {test_text}")
print(f"Translated: {result}")

Original: Hello, world!
Translated: Привет, мир!


In [40]:
# Настройки
SOURCE_REPO_ID = "DeepPavlov/coqa_abg"
LOCAL_SAVE_PATH = "./coqa_abg_ru"
CACHE_FILE = "translation_cache_coqa.jsonl"
SPLITS = ['train', 'val', 'test']  # в датасете validation вместо dev

In [41]:
# ---------- Кэш переводов ----------
def load_cache():
    cache = {}
    if not os.path.exists(CACHE_FILE):
        return cache
    try:
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except:
                    continue
    except Exception as e:
        logging.error(f"Failed to load cache: {e}")
    return cache

def append_cache(text, translation):
    try:
        with open(CACHE_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")
    except Exception as e:
        logging.error(f"Failed to append cache: {e}")

translation_cache = load_cache()

import time
import logging

def translate_text(text, retries=3, delay=3):
    """
    Переводит текст через Яндекс.Переводчик.
    - Для коротких числовых строк (только цифры, длина ≤ 4) возвращает оригинал без вызова API.
    - Использует кэш.
    - При ошибках делает до 3 попыток с фиксированной задержкой 3 секунды.
    """
    if not isinstance(text, str) or text.strip() == "":
        return "", True

    # Числовые строки до 4 символов оставляем без перевода
    if text.strip().isdigit() and len(text.strip()) <= 4:
        return text, True

    # Проверка кэша
    if text in translation_cache:
        return translation_cache[text], True

    last_exception = None
    for attempt in range(retries):
        try:
            # Фиксированная задержка перед каждым запросом (включая первый)
            if attempt > 0:
                time.sleep(delay)
            else:
                time.sleep(0.5)  # небольшая пауза перед первым запросом

            result = yandex.translate(text, "ru")
            translated = str(result.result) if hasattr(result, 'result') else str(result)

            translation_cache[text] = translated
            append_cache(text, translated)
            return translated, True

        except Exception as e:
            last_exception = e
            error_str = str(e).lower()

            # Определяем тип ошибки для информативного лога
            if any(code in error_str for code in ['502', '503', '504']):
                logging.warning(f"Server error (5xx) for '{text[:30]}...'. "
                                f"Attempt {attempt+1}/{retries}.")
            elif '429' in error_str or 'too many requests' in error_str:
                logging.warning(f"Rate limit hit. Attempt {attempt+1}/{retries}.")
            else:
                logging.warning(f"Translation error for '{text[:30]}...'. "
                                f"Attempt {attempt+1}/{retries}. Error: {e}")

            # Ждём фиксированное время перед следующей попыткой
            if attempt < retries - 1:
                time.sleep(delay)

    logging.error(f"Failed to translate after {retries} attempts: '{text[:50]}...' Error: {last_exception}")
    return "", False

def translate_example(example):
    success_overall = True

    # 1. Story
    story_ru, ok = translate_text(example['story'])
    success_overall = success_overall and ok

    # 2. target_turn – переводим question, answer, rationale
    target = example['target_turn']
    q_ru, ok_q = translate_text(target['question'])
    a_ru, ok_a = translate_text(target['answer'])
    r_ru, ok_r = translate_text(target['rationale'])
    success_overall = success_overall and ok_q and ok_a and ok_r

    target_ru = {
        'turn_id': target['turn_id'],
        'span_start': target['span_start'],
        'span_end': target['span_end'],
        'rationale_ru': r_ru,
        'question_ru': q_ru,
        'answer_ru': a_ru
    }

    # 3. history_turns – переводим каждый turn (question, answer, rationale)
    history_ru = []
    for turn in example['history_turns']:
        q_ru, ok_q = translate_text(turn['question'])
        a_ru, ok_a = translate_text(turn['answer'])
        r_ru, ok_r = translate_text(turn['rationale'])
        success_overall = success_overall and ok_q and ok_a and ok_r

        turn_ru = {
            'turn_id': turn['turn_id'],
            'rationale_ru': r_ru,
            'question_ru': q_ru,
            'answer_ru': a_ru
        }
        history_ru.append(turn_ru)

    # 4. Остальные поля без изменений
    return {
        'id': example['id'],
        'story': example['story'],
        'story_ru': story_ru,
        'target_turn': target,
        'target_turn_ru': target_ru,
        'history_turns': example['history_turns'],
        'history_turns_ru': history_ru,
        'ambiguity': example.get('ambiguity', ''),
        'clarification_turn': example.get('clarification_turn'),
        'source': example.get('source', ''),
        'clarification_turn_2': example.get('clarification_turn_2'),
        '_success': success_overall
    }

In [42]:
# ---------- Обработка сплита с прогрессом ----------
def process_split(split_name, source_split, progress_file):
    translated_records = []
    failed_indices = set()

    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if rec.get('_failed', False):
                        failed_indices.add(rec['_index'])
                    else:
                        translated_records.append(rec)
                except:
                    continue
        logging.info(f"[{split_name}] Resuming. Found {len(translated_records)} successful, {len(failed_indices)} failed.")

    start_index = len(translated_records) + len(failed_indices)
    total = len(source_split)

    if start_index < total:
        logging.info(f"[{split_name}] Starting translation from index {start_index}...")
        with open(progress_file, "a", encoding="utf-8") as f:
            pbar = tqdm(
                enumerate(source_split.select(range(start_index, total))),
                desc=f"Translating {split_name}",
                total=total - start_index
            )
            for idx, example in pbar:
                global_idx = start_index + idx
                if global_idx in {r['_index'] for r in translated_records}:
                    continue

                translated = translate_example(example)
                record_out = {
                    '_index': global_idx,
                    '_failed': not translated['_success'],
                    **translated
                }
                del record_out['_success']

                f.write(json.dumps(record_out, ensure_ascii=False) + "\n")
                f.flush()
                time.sleep(0.5)

                if not record_out['_failed']:
                    translated_records.append(record_out)
                else:
                    failed_indices.add(global_idx)

    # Сбор успешных записей
    all_successful = []
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if not rec.get('_failed', False):
                        rec.pop('_index', None)
                        rec.pop('_failed', None)
                        all_successful.append(rec)
                except:
                    continue

    if not all_successful:
        logging.error(f"[{split_name}] No records successfully translated.")
        return None

    total_orig = len(source_split)
    ok_cnt = len(all_successful)
    fail_cnt = total_orig - ok_cnt
    logging.info(f"[{split_name}] Translation completed: {ok_cnt}/{total_orig} successful ({fail_cnt} failed)")
    if fail_cnt > 0:
        logging.info(f"[{split_name}] To retry failed, delete/modify {progress_file} and run again.")

    return Dataset.from_list(all_successful)

In [10]:
# ---------- Запуск ----------
logging.info(f"Loading dataset '{SOURCE_REPO_ID}'...")
source = load_dataset(SOURCE_REPO_ID)

2026-04-15 21:15:25,659 - INFO - Loading dataset 'DeepPavlov/coqa_abg'...


In [ ]:
translated_splits = {}
for split in SPLITS:
    if split not in source:
        logging.warning(f"Split '{split}' not found, skipping...")
        continue
    progress_path = f"translated_coqa_{split}.jsonl"
    ds_translated = process_split(split, source[split], progress_path)
    if ds_translated is None:
        logging.error(f"Failed to process split {split}")
        break
    translated_splits[split] = ds_translated

if len(translated_splits) == len(SPLITS):
    final_dataset = DatasetDict(translated_splits)
    logging.info(f"Saving dataset to '{LOCAL_SAVE_PATH}'...")
    final_dataset.save_to_disk(LOCAL_SAVE_PATH)
    print(f"\nDataset saved to {LOCAL_SAVE_PATH}")

    print("\n" + "="*60)
    print("FINAL TRANSLATED DATASET")
    print("="*60)
    for split_name, ds in final_dataset.items():
        print(f"{split_name}: {len(ds)} examples, features: {ds.column_names}")

    ex = final_dataset['train'][0]
    print("\n--- EXAMPLE (train) ---")
    print(f"Story original: {ex['story'][:150]}...")
    print(f"Story RU     : {ex['story_ru'][:150]}...")
    print(f"\nTarget turn:")
    print(f"  Q: {ex['target_turn']['question']}")
    print(f"  Q_ru: {ex['target_turn_ru']['question_ru']}")
    print(f"  A: {ex['target_turn']['answer']}")
    print(f"  A_ru: {ex['target_turn_ru']['answer_ru']}")
    if ex['history_turns']:
        ht = ex['history_turns'][0]
        ht_ru = ex['history_turns_ru'][0]
        print(f"\nHistory turn 0:")
        print(f"  Q: {ht['question']} -> {ht_ru['question_ru']}")
        print(f"  A: {ht['answer']} -> {ht_ru['answer_ru']}")
else:
    logging.error("Not all splits were successfully translated.")

In [ ]:
from datasets import DatasetDict, load_from_disk
from huggingface_hub import login

# --- Авторизация ---
login(token="YOUR_HF_TOKEN")  # замените на ваш токен

# --- Загружаем локальный датасет ---
LOCAL_PATH = "./coqa_abg_ru_v2"
REPO_ID = "DeepPavlov/coqa_abg_ru"

dataset = DatasetDict.load_from_disk(LOCAL_PATH)

# --- Загружаем каждый сплит как отдельную конфигурацию ---
for split_name, ds in dataset.items():
    # Имя конфигурации: train, validation, test
    config_name = split_name  # оставляем как есть
    ds.push_to_hub(
        REPO_ID,
        config_name=config_name,
        private=False,
        commit_message=f"Upload {config_name} split with Russian translations"
    )
    print(f"Загружен конфиг: {config_name}")

print(f"Готово! Датасет доступен по адресу: https://huggingface.co/datasets/{REPO_ID}")

Переводим еще раз то, что упало с ошибками

In [ ]:
import json

CACHE_FILE = "translation_cache_coqa.jsonl"

empty_or_bad = []
total = 0
with open(CACHE_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        total += 1
        obj = json.loads(line)
        text = obj.get('text', '')
        translation = obj.get('translation', '')
        # Проверяем: пустой перевод или содержит явные признаки ошибки
        if not translation or translation == text or len(translation) < 3:
            empty_or_bad.append((text, translation))

print(f"Всего записей в кэше: {total}")
print(f"Подозрительных (пустых или плохих): {len(empty_or_bad)}")
if empty_or_bad:
    print("\nПримеры проблемных записей (первые 10):")
    for text, trans in empty_or_bad[:10]:
        print(f"  '{text[:60]}...' -> '{trans[:60]}...'")

In [ ]:
import os, re, json, time, logging, shutil
from datasets import load_dataset, Dataset, DatasetDict
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# -------------------- Настройки --------------------
yandex = YandexTranslate()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

SOURCE_REPO_ID = "DeepPavlov/coqa_abg"
OLD_CACHE_FILE = "translation_cache_coqa.jsonl"
NEW_CACHE_FILE = "translation_cache_coqa_fixed.jsonl"
PROGRESS_DIR = "translated_cache_old"
SPLITS = ['train', 'val', 'test']
LOCAL_SAVE_PATH = "./coqa_abg_ru_final"

# ---------- Копируем старый кэш в новый ----------
if os.path.exists(OLD_CACHE_FILE):
    shutil.copy(OLD_CACHE_FILE, NEW_CACHE_FILE)
    print(f"Старый кэш скопирован в {NEW_CACHE_FILE}")
else:
    print("Старый кэш не найден, создаём новый с нуля")

CACHE_FILE = NEW_CACHE_FILE

# ---------- Кэш ----------
def load_cache():
    cache = {}
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except:
                    continue
    return cache

def append_cache(text, translation):
    with open(CACHE_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")

translation_cache = load_cache()

# ---------- Улучшенный переводчик ----------
def is_numeric_string(s: str) -> bool:
    return not bool(re.search(r'[A-Za-zА-Яа-яёЁ]', s))

def translate_text_robust(text, retries=3, delay=3):
    if not isinstance(text, str) or text.strip() == "":
        return "", True
    if is_numeric_string(text):
        return text, True
    if text in translation_cache:
        return translation_cache[text], True

    last_exception = None
    for attempt in range(retries):
        try:
            time.sleep(0.5 if attempt == 0 else delay)
            result = yandex.translate(text, "ru")
            translated = str(result.result) if hasattr(result, 'result') else str(result)
            translation_cache[text] = translated
            append_cache(text, translated)
            return translated, True
        except Exception as e:
            last_exception = e
            error_str = str(e).lower()
            if any(code in error_str for code in ['502', '503', '504']):
                wait = delay * (attempt + 1)
                logging.warning(f"Server error '{text[:30]}...' attempt {attempt+1}/{retries}. Wait {wait}s")
                time.sleep(wait)
            elif '429' in error_str:
                wait = delay * 4 + 10
                logging.warning(f"Rate limit. Wait {wait}s")
                time.sleep(wait)
            else:
                time.sleep(delay)

    if last_exception and '502' in str(last_exception).lower() and len(text) > 2000:
        logging.info(f"Long text failed, splitting: '{text[:30]}...'")
        sentences = re.split(r'(?<=[.!?])\s+', text)
        if len(sentences) > 1:
            parts = []
            for sent in sentences:
                part, ok = translate_text_robust(sent)
                if not ok:
                    break
                parts.append(part)
            if len(parts) == len(sentences):
                full = ' '.join(parts)
                translation_cache[text] = full
                append_cache(text, full)
                return full, True

    logging.error(f"Failed to translate: '{text[:50]}...' Error: {last_exception}")
    return "", False

# ---------- Перевод одного примера ----------
def translate_example_robust(example):
    success = True
    story_ru, ok = translate_text_robust(example['story'])
    success = success and ok

    target = example['target_turn']
    q_ru, ok_q = translate_text_robust(target['question'])
    a_ru, ok_a = translate_text_robust(target['answer'])
    r_ru, ok_r = translate_text_robust(target['rationale'])
    success = success and ok_q and ok_a and ok_r

    target_ru = {
        'turn_id': target['turn_id'],
        'span_start': target['span_start'],
        'span_end': target['span_end'],
        'rationale_ru': r_ru,
        'question_ru': q_ru,
        'answer_ru': a_ru
    }

    history_ru = []
    for turn in example['history_turns']:
        q_ru, ok_q = translate_text_robust(turn['question'])
        a_ru, ok_a = translate_text_robust(turn['answer'])
        r_ru, ok_r = translate_text_robust(turn['rationale'])
        success = success and ok_q and ok_a and ok_r
        history_ru.append({
            'turn_id': turn['turn_id'],
            'rationale_ru': r_ru,
            'question_ru': q_ru,
            'answer_ru': a_ru
        })

    return {
        'id': example['id'],
        'story': example['story'],
        'story_ru': story_ru,
        'target_turn': target,
        'target_turn_ru': target_ru,
        'history_turns': example['history_turns'],
        'history_turns_ru': history_ru,
        'ambiguity': example.get('ambiguity', ''),
        'clarification_turn': example.get('clarification_turn'),
        'source': example.get('source', ''),
        'clarification_turn_2': example.get('clarification_turn_2'),
        '_success': success
    }

# ---------- Обработка упавших из старых файлов ----------
def fix_failed_records(split_name, old_progress_file, new_progress_file, source_split):
    records = []
    with open(old_progress_file, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                records.append(json.loads(line))
            except:
                continue

    total_failed = sum(1 for r in records if r.get('_failed', False))
    logging.info(f"[{split_name}] Found {total_failed} failed records to redo.")
    if total_failed == 0:
        logging.info(f"[{split_name}] No failed records, nothing to do.")
        return

    fixed = 0
    with open(new_progress_file, 'w', encoding='utf-8') as out_f:
        for record in tqdm(records, desc=f"Fixing {split_name}"):
            if not record.get('_failed', False):
                out_f.write(json.dumps(record, ensure_ascii=False) + '\n')
                continue

            idx = record['_index']
            orig_example = source_split[int(idx)]
            translated = translate_example_robust(orig_example)
            success = translated['_success']
            del translated['_success']

            new_record = {
                '_index': idx,
                '_failed': not success,
                **translated
            }
            out_f.write(json.dumps(new_record, ensure_ascii=False) + '\n')
            if success:
                fixed += 1

    logging.info(f"[{split_name}] Fixed {fixed}/{total_failed} failed records. New file: {new_progress_file}")

# ---------- Сборка датасета из нового прогресс-файла ----------
def build_dataset_from_progress(progress_file):
    clean = []
    with open(progress_file, 'r', encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            if not rec.get('_failed', False):
                rec.pop('_index', None)
                rec.pop('_failed', None)
                clean.append(rec)
    return Dataset.from_list(clean)

# ---------- Запуск ----------
source = load_dataset(SOURCE_REPO_ID)

final_splits = {}
for split in SPLITS:
    old_file = os.path.join(PROGRESS_DIR, f"translated_coqa_{split}.jsonl")
    new_file = f"translated_coqa_{split}_fixed.jsonl"

    if not os.path.exists(old_file):
        logging.warning(f"Old file {old_file} not found, skipping {split}")
        continue

    fix_failed_records(split, old_file, new_file, source[split])
    final_splits[split] = build_dataset_from_progress(new_file)
    print(f"{split}: {len(final_splits[split])} good records")

final_dataset = DatasetDict(final_splits)
final_dataset.save_to_disk(LOCAL_SAVE_PATH)
print(f"Финальный датасет сохранён в {LOCAL_SAVE_PATH}")
print(f"Новый кэш: {NEW_CACHE_FILE}")

In [ ]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")

REPO_ID = "DeepPavlov/coqa_abg_ru"
final_dataset = DatasetDict.load_from_disk("./coqa_abg_ru_final")

for split_name, ds in final_dataset.items():
    ds.push_to_hub(
        REPO_ID,
        config_name=split_name,
        private=False,
        commit_message="All failed records fixed and retranslated"
    )
    print(f"{split_name} uploaded")

print("Готово: https://huggingface.co/datasets/DeepPavlov/coqa_abg_ru")